# Treinamento Python — Nível 6 — Módulos e Arquivos

> **Data Engineering Track** | Python do zero ao Data Engineering
> Cada seção termina com um **exercício de fixação** e ao final há um **desafio integrador**.

---
# 📘 Aula 01 Modulos E Pacotes

## NÍVEL 6 — Módulos e Arquivos | Aula 1: Módulos e Pacotes

## 1. Importando módulos da biblioteca padrão

In [ ]:
import os
import sys
import math
import random
import datetime
from pathlib import Path
from collections import defaultdict, Counter, OrderedDict
from itertools import chain, groupby

# math
print(math.pi)
print(math.sqrt(144))    # 12.0
print(math.ceil(4.3))    # 5
print(math.floor(4.9))   # 4

# random
print(random.randint(1, 10))
lista = [1, 2, 3, 4, 5]
random.shuffle(lista)
print(lista)
print(random.choice(lista))
random.seed(42)   # para resultados reproduzíveis

# datetime
hoje = datetime.date.today()
agora = datetime.datetime.now()
print(f"Data: {hoje}")
print(f"Data/hora: {agora.strftime('%d/%m/%Y %H:%M')}")

delta = datetime.timedelta(days=30)
print(f"Daqui 30 dias: {hoje + delta}")

## 2. collections — estruturas avançadas

defaultdict: dicionário com valor padrão para chaves novas

In [ ]:
vendas_por_regiao = defaultdict(list)
registros = [("Sul", 500), ("Norte", 300), ("Sul", 750), ("Norte", 420)]
for regiao, valor in registros:
    vendas_por_regiao[regiao].append(valor)
print(dict(vendas_por_regiao))

# Counter: conta ocorrências
palavras = ["python", "data", "python", "engineer", "data", "python"]
contagem = Counter(palavras)
print(contagem)
print(contagem.most_common(2))   # [('python', 3), ('data', 2)]

## 3. os e pathlib — sistema de arquivos

pathlib é a forma moderna (Python 3.4+)

In [ ]:
caminho = Path("/home/lg/Documents/personal/projects/treinamento/python")

print(caminho.exists())
print(caminho.is_dir())
print(caminho.name)       # nome final do caminho
print(caminho.parent)     # diretório pai

# Listando arquivos
for arquivo in sorted(caminho.rglob("*.py")):
    print(f"  {arquivo.relative_to(caminho)}")

# Criando diretórios
novo_dir = caminho / "nivel_6" / "dados"
novo_dir.mkdir(parents=True, exist_ok=True)

# Informações do arquivo
py_files = list(caminho.rglob("*.py"))
if py_files:
    f = py_files[0]
    print(f"\nArquivo: {f.name}")
    print(f"Tamanho: {f.stat().st_size} bytes")
    print(f"Sufixo: {f.suffix}")
    print(f"Stem: {f.stem}")

# os.environ — variáveis de ambiente
home = os.environ.get("HOME", "/tmp")
print(f"HOME: {home}")

## 4. Formas de importação

import modulo                → modulo.funcao()
from modulo import funcao    → funcao()
from modulo import *         → EVITE — polui o namespace
import modulo as alias       → alias.funcao()
from modulo import funcao as alias

In [ ]:
from datetime import datetime as dt
from pathlib import Path as P

print(dt.now().year)
print(P(".").resolve())

## EXERCÍCIO DE FIXAÇÃO 6.1

Use os módulos da biblioteca padrão para:
  1. Listar todos os arquivos .py criados no treinamento
  2. Contar quantos arquivos existem por nível (nivel_1, nivel_2, etc.)
  3. Exibir o arquivo mais recente por data de modificação

In [ ]:
# Escreva seu código aqui


---
# 📘 Aula 02 Leitura E Escrita Arquivos

## NÍVEL 6 — Módulos e Arquivos | Aula 2: Leitura e Escrita

In [ ]:
import csv
import json
from pathlib import Path

BASE = Path("/home/lg/Documents/personal/projects/treinamento/python/nivel_6/dados")
BASE.mkdir(parents=True, exist_ok=True)

## 1. Arquivos de texto (.txt)

In [ ]:
arquivo_txt = BASE / "log.txt"

# Escrevendo
with open(arquivo_txt, "w", encoding="utf-8") as f:
    f.write("2026-05-01 08:00 | INFO  | Pipeline iniciado\n")
    f.write("2026-05-01 08:01 | INFO  | 1500 registros extraídos\n")
    f.write("2026-05-01 08:03 | WARN  | 12 registros nulos ignorados\n")
    f.write("2026-05-01 08:05 | INFO  | Pipeline finalizado com sucesso\n")

# Lendo tudo
with open(arquivo_txt, "r", encoding="utf-8") as f:
    conteudo = f.read()
    print("=== Conteúdo completo ===")
    print(conteudo)

# Lendo linha a linha (eficiente para arquivos grandes)
with open(arquivo_txt, "r", encoding="utf-8") as f:
    for linha in f:
        partes = linha.strip().split(" | ")
        data, nivel, msg = partes
        print(f"[{nivel}] {data}: {msg}")

# Adicionando ao arquivo (modo append)
with open(arquivo_txt, "a", encoding="utf-8") as f:
    f.write("2026-05-01 08:06 | INFO  | Relatório gerado\n")

## 2. CSV

In [ ]:
arquivo_csv = BASE / "vendas.csv"

# Escrevendo CSV
vendas = [
    {"data": "2026-05-01", "produto": "Arroz",  "qtd": 10, "valor": 185.00},
    {"data": "2026-05-01", "produto": "Feijão", "qtd": 25, "valor": 197.50},
    {"data": "2026-05-02", "produto": "Arroz",  "qtd": 8,  "valor": 148.00},
    {"data": "2026-05-02", "produto": "Milho",  "qtd": 15, "valor": 48.00},
]

with open(arquivo_csv, "w", newline="", encoding="utf-8") as f:
    campos = ["data", "produto", "qtd", "valor"]
    writer = csv.DictWriter(f, fieldnames=campos)
    writer.writeheader()
    writer.writerows(vendas)

print("\n=== CSV escrito ===")

# Lendo CSV
with open(arquivo_csv, "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    total = 0
    for row in reader:
        valor = float(row["valor"])
        total += valor
        print(f"  {row['data']} | {row['produto']:<10} | {row['qtd']:>3} un | R${valor:.2f}")
    print(f"  Total: R${total:.2f}")

## 3. JSON

In [ ]:
arquivo_json = BASE / "config.json"

# Escrevendo JSON
config = {
    "pipeline": {
        "nome": "ETL Vendas Diário",
        "versao": "2.1.0",
        "ativo": True
    },
    "fontes": [
        {"tipo": "postgres", "host": "db.empresa.com", "porta": 5432},
        {"tipo": "csv", "caminho": "/data/vendas/*.csv"}
    ],
    "notificacoes": {
        "email": ["equipe@empresa.com"],
        "slack": "#data-alerts"
    }
}

with open(arquivo_json, "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2, ensure_ascii=False)

print("\n=== JSON escrito ===")

# Lendo JSON
with open(arquivo_json, "r", encoding="utf-8") as f:
    dados = json.load(f)

print(f"Pipeline: {dados['pipeline']['nome']} v{dados['pipeline']['versao']}")
print(f"Fontes: {len(dados['fontes'])}")
for fonte in dados['fontes']:
    print(f"  - {fonte['tipo']}")

# JSON em memória (sem arquivo)
json_string = '{"nome": "Ana", "idade": 28}'
obj = json.loads(json_string)          # string → dict
print(json.dumps(obj, indent=2))       # dict → string

## EXERCÍCIO DE FIXAÇÃO 6.2

Leia o CSV de vendas criado acima e:
  1. Agrupe o total vendido por produto
  2. Salve o resultado em um JSON
  3. Leia o JSON e imprima um relatório formatado

In [ ]:
# Escreva seu código aqui


---
# 🏆 Desafio Nivel 6

## NÍVEL 6 — DESAFIO FINAL | Pipeline ETL com Arquivos Reais

CONTEXTO:
Você receberá dados de vendas em CSV (simulado),
fará transformações e salvará os resultados em JSON.
O sistema também gerará um log de execução em .txt.

REQUISITOS:
1. Gerar dados brutos em CSV (simular extração)
2. Ler e transformar os dados (limpeza, tipagem, cálculos)
3. Agrupar por categoria e calcular estatísticas
4. Salvar resultado final em JSON
5. Registrar todo o processo em arquivo de log
6. Exibir relatório final

In [ ]:
# Escreva seu código aqui
